In [2]:
from pyqmt.service.qmt_broker import QMTBroker

from pyqmt.config import cfg
import cfg4py
import os

os.environ["__cfg4py_server_role__"] = "DEV"

cfg4py.init()

broker = QMTBroker(cfg)
broker.asset


c:\Users\aaron\miniconda3\envs\qmt\Lib\site-packages\xtquant\__init__.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution


ImportError: cannot import name 'TradeData' from 'pyqmt.core.enums' (C:\Users\aaron\workspace\pyqmt\pyqmt\core\enums.py)

In [6]:
order.status_msg = "cancelled"

db.update_order(order)
db.get_order(order.oid)

OrderModel(asset='symbol.SZ', side='1', shares='1000', price=100.0, bid_type=1, tm=None, fid='456', cid='789', status='49', status_msg='cancelled', oid='fd777ea3-4dc0-485a-95ba-da32aa39d3ad', strategy='', remark='')

In [14]:
from pyqmt.dal.tradedb import db 
import tempfile 
import os

# 创建临时数据库文件 
temp_db_path = tempfile.mktemp(suffix='.db')

# 手动初始化，不使用WAL模式
import sqlite3
import sqlite_utils as su
from pyqmt.models import OrderModel, TradeModel, AssetModel, PositionModel

# 创建连接并初始化
conn = sqlite3.connect(temp_db_path)
db_obj = su.Database(conn)

# 手动创建表结构
for model in [OrderModel, TradeModel, AssetModel, PositionModel]:
    table = model.__table_name__
    pk = model.__pk__
    
    t = db_obj[table]
    t.create(model.to_db_schema(), pk=pk)
    
    # 创建索引
    if model.__indexes__ is not None:
        indexes, is_unique = model.__indexes__
        t.create_index(indexes, unique=is_unique)
    
    # 创建外键约束
    if hasattr(model, '__foreign_keys__') and model.__foreign_keys__:
        for fk in model.__foreign_keys__:
            if len(fk) == 3:  # (from_column, to_table, to_column)
                from_col, to_table, to_col = fk
                t.add_foreign_key(from_col, to_table, to_col)

# 提交并关闭
conn.commit()
print("Tables created successfully:", list(db_obj.tables))
conn.close()

# 现在使用TradeDB初始化
db.init(temp_db_path)
current_db = db.db
print("TradeDB tables:", list(current_db.tables))

Tables created successfully: [<Table orders (asset, side, shares, price, bid_type, tm, foid, cid, status, status_msg, qtoid, strategy)>, <Table trades (tid, qtoid, foid, asset, shares, price, amount, tm, side, cid, fee)>, <Table assets (dt, principal, cash, frozen_cash, market_value, total)>, <Table positions (dt, asset, shares, avail, price)>]
TradeDB tables: []


In [1]:
from pathlib import Path
import tempfile
from pyqmt.dal.tradedb import db
from pyqmt.models import AssetModel
from dataclasses import asdict
import datetime

_, file = tempfile.mkstemp()
db._initialized = False
db.init(file)

db.db["assets"].columns

[Column(cid=0, name='dt', type='TEXT', notnull=0, default_value=None, is_pk=1),
 Column(cid=1, name='principal', type='FLOAT', notnull=0, default_value=None, is_pk=0),
 Column(cid=2, name='cash', type='FLOAT', notnull=0, default_value=None, is_pk=0),
 Column(cid=3, name='frozen_cash', type='FLOAT', notnull=0, default_value=None, is_pk=0),
 Column(cid=4, name='market_value', type='FLOAT', notnull=0, default_value=None, is_pk=0),
 Column(cid=5, name='total', type='FLOAT', notnull=0, default_value=None, is_pk=0)]

In [2]:
asset = AssetModel(datetime.datetime.today(),
                    1_000_000,
                    1_000_000,
                    0,
                    0,
                    1_000_000)
db.db["assets"].insert(asdict(asset))

<Table assets (dt, principal, cash, frozen_cash, market_value, total)>

In [3]:
model = AssetModel(**list(db.db["assets"].rows)[0])
model.dt

datetime.date(2025, 12, 25)

In [4]:
db.query_asset_by_date(datetime.date(2025, 12, 25))

AssetModel(dt=datetime.date(2025, 12, 25), principal=1000000.0, cash=1000000.0, frozen_cash=0.0, market_value=0.0, total=1000000.0)

In [23]:
db.db["assets"].drop()

In [22]:
datetime.date.today()

datetime.date(2025, 12, 25)

In [107]:
import time

# 1. 记录开始时间
start = time.perf_counter()

# 2. 执行需要计时的代码（示例：循环/函数调用）
total = 0
for i in range(1000000):
    total += i

# 3. 记录结束时间，计算耗时
end = time.perf_counter()
elapsed = end - start

# 4. 格式化输出（微秒/毫秒/秒）
print(f"耗时：{elapsed:.6f} 秒")          # 保留6位小数（微秒级）
print(f"耗时：{elapsed * 1000:.3f} 毫秒")  # 毫秒级
print(f"耗时：{elapsed * 1e6:.0f} 微秒")   # 微秒级

耗时：0.168806 秒
耗时：168.806 毫秒
耗时：168806 微秒
